In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Load data

CSV_PATH = "listings.csv" 

def _clean_money_to_float(s: pd.Series) -> pd.Series:
    # "$136.00" -> 136.00
    return (
        s.astype(str)
         .str.replace(r"[\$,]", "", regex=True)
         .replace("nan", np.nan)
         .astype(float)
    )

# Targets in your dataset

TARGET_PRICE = "price"
TARGET_RATE  = "estimated_occupancy_l365d"  

# Feature columns (these EXIST in your dataset)

categorical_cols = [
    "room_type",
    "property_type",
    "neighbourhood",
    "neighbourhood_group_cleansed"
]

numeric_cols = [
    "accommodates", "bedrooms", "beds", "bathrooms",
    "latitude", "longitude",
    "minimum_nights", "maximum_nights",
    "availability_30", "availability_60", "availability_90", "availability_365",
    "number_of_reviews", "reviews_per_month",
    "review_scores_rating", "review_scores_cleanliness", "review_scores_location"
]

FEATURE_COLS = numeric_cols + categorical_cols

# Preprocessing (with StandardScaler)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ],
    remainder="drop"
)

# Models 

def _make_model():
    return RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

price_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", _make_model())
])

rate_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", _make_model())
])


# Train models

def train_models(csv_path=CSV_PATH):
    df = pd.read_csv(csv_path)

    # Clean price to numeric
    df[TARGET_PRICE] = _clean_money_to_float(df[TARGET_PRICE])

    # ---- PRICE MODEL ----
    df_price = df.dropna(subset=[TARGET_PRICE]).copy()
    Xp = df_price[FEATURE_COLS]
    yp = df_price[TARGET_PRICE]

    Xp_train, Xp_test, yp_train, yp_test = train_test_split(
        Xp, yp, test_size=0.2, random_state=42
    )
    price_pipeline.fit(Xp_train, yp_train)
    pred_p = price_pipeline.predict(Xp_test)
    print("PRICE MODEL | MAE:", round(mean_absolute_error(yp_test, pred_p), 2),
          "| R2:", round(r2_score(yp_test, pred_p), 3))

    # ---- RATE MODEL ----
    df_rate = df.dropna(subset=[TARGET_RATE]).copy()
    Xr = df_rate[FEATURE_COLS]
    yr = df_rate[TARGET_RATE]

    Xr_train, Xr_test, yr_train, yr_test = train_test_split(
        Xr, yr, test_size=0.2, random_state=42
    )
    rate_pipeline.fit(Xr_train, yr_train)
    pred_r = rate_pipeline.predict(Xr_test)
    print("RATE MODEL  | MAE:", round(mean_absolute_error(yr_test, pred_r), 2),
          "| R2:", round(r2_score(yr_test, pred_r), 3))

    return price_pipeline, rate_pipeline

PRICE_MODEL, RATE_MODEL = train_models(CSV_PATH)


# 6) Prediction helpers (use in Dash callbacks)

def predict_price(input_dict: dict) -> float:
    X = pd.DataFrame([input_dict])[FEATURE_COLS]
    return float(PRICE_MODEL.predict(X)[0])

def predict_rate(input_dict: dict) -> float:
    X = pd.DataFrame([input_dict])[FEATURE_COLS]
    return float(RATE_MODEL.predict(X)[0])


/opt/anaconda3/lib/python3.13/site-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['neighbourhood_group_cleansed']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['neighbourhood_group_cleansed']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['neighbourhood_group_cleansed']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(


PRICE MODEL | MAE: 62.45 | R2: 0.676
RATE MODEL  | MAE: 29.57 | R2: 0.738


/opt/anaconda3/lib/python3.13/site-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['neighbourhood_group_cleansed']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
